[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/08_alumno_hiperparametros.ipynb)

# MLY1101 · Machine Learning — Actividad 3.1
## Ajuste de hiperparámetros

**Resultado de aprendizaje (RA3):** elabora soluciones avanzadas de aprendizaje automático
mediante la optimización de hiperparámetros, técnicas de ensamble y validación cruzada, para
garantizar la precisión y generalización del modelo frente a objetivos de negocio complejos.

**Indicador de logro (IL 3.1):** aplica estrategias de ajuste de hiperparámetros para maximizar
el rendimiento y la eficiencia de los modelos seleccionados.

---

### Dónde estamos

El RA2 dejó un modelo que funciona: F1-macro de **0,70**, con un recall de 0,40 en la clase
minoritaria. La pregunta del RA3 es si se puede hacer mejor, y **cómo saber si de verdad
mejoró**.

Hoy: los hiperparámetros. Los parámetros que el modelo **no** aprende de los datos y que hay
que elegir desde fuera: cuántos árboles, qué profundidad, cuántas muestras por hoja.

---

### La idea central, y probablemente te va a decepcionar

> **El ajuste de hiperparámetros da mejoras de segundo orden.**

Las mejoras de primer orden vienen de otra parte: de las variables que elegiste, de cómo
partiste los datos, de haber definido bien el problema. Si el modelo va mal, ajustar
hiperparámetros casi nunca lo salva.

Al final de la sesión vas a haber probado 12 configuraciones distintas y vas a comparar la
mejor contra los valores por defecto. **Guarda tu expectativa** de cuánto vas a ganar.

---

### Dónde se ajusta, que es lo que de verdad se evalúa

Ajustar exige comparar configuraciones, y comparar exige medir. ¿Medir dónde?

- **En entrenamiento:** no sirve. Más complejidad siempre puntúa mejor ahí.
- **En prueba:** es hacer trampa. Si eliges la configuración que mejor puntúa en la prueba, esa
  puntuación deja de estimar el futuro.
- **En validación cruzada dentro del entrenamiento:** correcto. Y aquí, por lo mismo del RA2,
  los pliegues tienen que respetar el segmento.

---

### Al final de la sesión debes entregar

El informe de ajuste: qué espacio exploraste, con qué esquema de validación, **cuánto ganaste**
y si esa ganancia supera la variabilidad entre pliegues.

---
## Preparación del entorno

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO.resolve()
else:
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "kedro_mly1101" / "src"))

RUTA_DATOS = RAIZ / "datos" / "crudos" / "detecciones_waymo_like.csv"
RUTA_PARAMETROS = RAIZ / "kedro_mly1101" / "conf" / "base" / "parameters.yml"
print("Colab:", EN_COLAB, "| dataset:", RUTA_DATOS.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from kedro_mly1101.pipelines.preprocesamiento import nodes as limpieza
from kedro_mly1101.pipelines.supervisado import nodes as supervisado
from kedro_mly1101.pipelines.optimizacion import nodes as optimizacion

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)
sns.set_theme(style="whitegrid")

PARAMETROS = yaml.safe_load(RUTA_PARAMETROS.read_text(encoding="utf-8"))
CONFIG, FUGA, AJUSTE = PARAMETROS["modelo"], PARAMETROS["fuga"], PARAMETROS["ajuste"]

# La misma cadena de siempre: limpieza del RA1 -> partición del RA2.
crudo = pd.read_csv(RUTA_DATOS)
paso = limpieza.normalizar_categorias(crudo, PARAMETROS["mapas_categorias"])
paso = limpieza.descubrir_faltantes(paso, PARAMETROS["centinelas"])
paso = limpieza.marcar_imposibles(paso, PARAMETROS["reglas_dominio"])
limpio = limpieza.quitar_duplicados_y_constantes(paso, PARAMETROS["columnas_a_descartar"])

marcada = supervisado.particionar(
    supervisado.preparar_variables(limpio, CONFIG, FUGA), CONFIG
)
entrena = marcada[marcada["particion"] == "entrenamiento"]

print(f"Entrenamiento: {len(entrena):,} filas en {entrena[CONFIG['grupo']].nunique()} segmentos")
print(f"Métrica de trabajo: {AJUSTE['metrica']}  ·  pliegues: {AJUSTE['n_pliegues']}")

---
# Bloque 1 · ⭐ Dónde se mide: validación cruzada por grupo

Para comparar configuraciones hace falta una estimación de desempeño **que no use la prueba**.
Se saca partiendo el entrenamiento en `k` pliegues: se entrena con `k−1` y se mide en el que
queda, `k` veces.

**Y los pliegues tienen que respetar el segmento**, exactamente por lo mismo que la partición
de la Actividad 2.2: las detecciones de un segmento comparten contexto. `GroupKFold` lo
garantiza.

### ✏️ TODO 1 — Comprobar que los pliegues no rompen segmentos

In [ ]:
# TODO 1: ¿los pliegues respetan el segmento?
from sklearn.model_selection import GroupKFold

X = entrena[CONFIG["variables"]]
y = entrena[CONFIG["objetivo"]]
grupos = entrena[CONFIG["grupo"]]

cv = ____(n_splits=AJUSTE["n_pliegues"])

filas = []
for numero, (idx_entrena, idx_valida) in enumerate(cv.split(X, y, groups=____), start=1):
    seg_entrena = set(grupos.iloc[idx_entrena])
    seg_valida = set(grupos.iloc[idx_valida])
    filas.append(
        {
            "pliegue": numero,
            "filas_entrena": len(idx_entrena),
            "filas_valida": len(idx_valida),
            "segmentos_valida": len(seg_valida),
            "segmentos_compartidos": len(seg_entrena & seg_valida),
        }
    )
pd.DataFrame(filas)

In [ ]:
# Autochequeo
tabla = pd.DataFrame(filas)
assert (tabla["segmentos_compartidos"] == 0).all(), (
    "revisa: ningún pliegue puede compartir segmentos. ¿Pasaste groups= al split?"
)
print(f"✅ {len(tabla)} pliegues, 0 segmentos compartidos en todos.")
print("   La estimación que salga de aquí es honesta: cada pliegue evalúa")
print("   sobre segmentos que el modelo nunca vio.")

### ✏️ TODO 2 — La métrica del ajuste

Antes de buscar hay que decidir **qué se está maximizando**. Mira el parámetro
`AJUSTE["metrica"]` y responde.

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

¿Por qué se optimiza `f1_macro` y no `accuracy`? *(Pista: revisa lo que descubriste en el
bloque 3 de la Actividad 2.2.)*

---
# Bloque 2 · Buscar: rejilla contra búsqueda aleatoria

| | Rejilla (`GridSearchCV`) | Aleatoria (`RandomizedSearchCV`) |
|---|---|---|
| Qué prueba | **Todas** las combinaciones | `n_iter` combinaciones al azar |
| Costo | Producto de las opciones: explota | El que tú decidas |
| Ventaja | Exhaustiva en su rejilla | Con el mismo presupuesto explora más regiones |

Con 4 × 6 × 4 × 3 = **288 combinaciones**, cada una con 5 pliegues, la rejilla exigiría 1.440
entrenamientos. La búsqueda aleatoria hace 12 × 5 = 60.

> **Por qué la aleatoria suele bastar:** casi siempre solo un par de hiperparámetros importan de
> verdad. La rejilla gasta la mayor parte del presupuesto variando los que dan igual.

### ✏️ TODO 3 — Lanzar la búsqueda

In [ ]:
# TODO 3: busca hiperparámetros con validación cruzada por grupo.
busqueda = optimizacion.____(marcada, CONFIG, AJUSTE)
busqueda.head(6)

### ✏️ TODO 4 — Leer la tabla con desconfianza

Mira las columnas `mean_test_score` y `std_test_score`.

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

Compara la diferencia entre la primera y la segunda configuración con la desviación típica
entre pliegues de la primera. ¿Puedes afirmar que la primera es mejor?

---
# Bloque 3 · ⭐⭐ ¿Cuánto ganamos de verdad?

Ya tenemos la mejor configuración de 12. Ahora la comparación que importa: **contra no haber
ajustado nada**.

### ✏️ TODO 5 — Antes de ejecutar, apuesta

**Creo que el ajuste mejorará el F1-macro en:** `____`

*(Escríbelo. Otra vez.)*

In [ ]:
# TODO 5: ¿cuánto ganó el ajuste sobre los valores por defecto?
ganancia = optimizacion.____(marcada, CONFIG, AJUSTE, busqueda)
ganancia

In [ ]:
# Autochequeo
delta = ganancia.loc[1, "ganancia"]
ruido = ganancia.loc[0, "desv_entre_pliegues"]
print(f"Ganancia del ajuste : {delta:+.4f}")
print(f"Ruido entre pliegues: {ruido:.4f}")
print()
assert abs(delta) < ruido, (
    "revisa: la ganancia debería quedar por debajo del ruido entre pliegues"
)
print("✅ La ganancia del ajuste es MENOR que la variabilidad entre pliegues.")
print("   Traducido: 12 configuraciones, 60 entrenamientos, y no hay evidencia")
print("   de haber mejorado nada.")

### ✏️ TODO 6 — Entonces, ¿el ajuste no sirve?

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. ¿Fue inútil esta sesión?
2. Si el ajuste da tan poco, ¿de dónde vienen las mejoras grandes en un proyecto de ML?
3. ¿En qué situación **sí** esperarías que el ajuste diera una mejora importante?

---
# Bloque 4 · ⭐ La trampa: ajustar mirando la prueba

Nadie *entrena* con la prueba. Pero mucha gente **elige** mirándola: prueba varias
configuraciones, ve cuál puntúa mejor en el conjunto de prueba y reporta ese número.

El modelo nunca vio esos datos. ¿Cuál es el problema?

Que la puntuación que reportas ya no es una estimación del desempeño futuro: es **el máximo de
una muestra**, y el máximo de una muestra siempre es optimista.

### ✏️ TODO 7 — Medirlo

In [ ]:
# TODO 7: ¿cuánto se infla la métrica al elegir mirando la prueba?
fuga = optimizacion.____(marcada, CONFIG, AJUSTE)
fuga

### ✏️ TODO 8 — Tres cosas distintas en una tabla

La tabla mide tres cosas que se confunden con facilidad. Explica cada una.

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. **Optimismo:** salió `____`. ¿Significa que la trampa es inofensiva?
2. **Margen de la trampa:** `____`. ¿Qué representa?
3. **Brecha validación vs prueba:** la validación cruzada puntúa sistemáticamente **más bajo**
   que la prueba. ¿Es esto una fuga? ¿Por qué pasa?

---
# Cierre · Informe de ajuste

**Modelo ajustado:** `____` · **Métrica optimizada:** `____` · **Por qué esa métrica:** `____`

### Esquema de validación

| Campo | Valor |
|---|---|
| Tipo de validación | `____` |
| Pliegues | `____` |
| Variable de agrupación | `____` |
| Segmentos compartidos entre pliegues | `____` |

### Espacio explorado

| Hiperparámetro | Valores | Por qué ese rango |
|---|---|---|
| `____` | | |
| `____` | | |

**Estrategia** (rejilla o aleatoria) **y por qué:** `____`
**Combinaciones probadas:** `____` de `____` posibles

### El resultado

| | F1-macro | Desv. entre pliegues |
|---|---|---|
| Valores por defecto | `____` | `____` |
| Mejor configuración | `____` | `____` |
| **Ganancia** | `____` | |

**¿La ganancia supera la variabilidad entre pliegues?** `____`
**Conclusión:** `____`

> Si tu ganancia no supera el ruido, **dilo**. Reportar una mejora que no puedes distinguir del
> azar es el error que esta sesión existe para evitar.

### Dónde buscaría la próxima mejora

`____`

*(Y por qué ahí y no en más ajuste.)*